In [36]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

In [37]:
# 1. Đọc dữ liệu từ file CSV gốc
df = pd.read_csv('../data/cleaned/cleaned_telco_churn.csv')

# Ép kiểu dữ liệu cho TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Gom tất cả các cột cần mã hóa (bao gồm cả cột nhị phân và nhiều nhóm)
categorical_cols = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
]
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

exclude_cols = ['customerID', 'Churn']

In [38]:
ohe = OneHotEncoder(sparse_output=False, dtype=int, drop=None)
ohe_array = ohe.fit_transform(df[categorical_cols])
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
df_ohe = pd.DataFrame(ohe_array, columns=ohe_feature_names, index=df.index)

# 4. Sử dụng StandardScaler cho các cột số
scaler = StandardScaler()
scaled_array = scaler.fit_transform(df[numeric_cols])
df_numeric = pd.DataFrame(scaled_array, columns=numeric_cols, index=df.index)\

# 5. Ghép các cột số đã chuẩn hóa và các cột đã One-Hot lại với nhau
df_final = pd.concat([df[exclude_cols],df_numeric, df_ohe], axis=1)

In [39]:
df_final.shape

(7032, 37)

In [40]:
df_final.head()

,customerID,Churn,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,...,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,No,-0.440327,-1.280248,-1.161694,-0.994194,0,1,0,1,...,1,0,0,1,0,0,0,0,1,0
1,5575-GNVDE,No,-0.440327,0.064303,-0.260878,-0.173740,1,0,0,1,...,1,0,0,0,1,0,0,0,0,1
2,3668-QPYBK,Yes,-0.440327,-1.239504,-0.363923,-0.959649,1,0,0,1,...,1,0,0,1,0,0,0,0,0,1
3,7795-CFOCW,No,-0.440327,0.512486,-0.747850,-0.195248,0,1,0,1,...,1,0,0,0,1,0,1,0,0,0
4,9237-HQITU,Yes,-0.440327,-1.239504,0.196178,-0.940457,1,0,0,0,...,1,0,0,1,0,0,0,0,1,0


In [41]:
# 6. Lưu toàn bộ tập dữ liệu thành file CSV vào thư mục đích
df_final.to_csv('../data/encoded/telco_churn_all_onehot.csv', index=False)

print("Đã chuẩn hóa, mã hóa và lưu thành công!")
print("Kích thước dữ liệu mới:", df_final.shape)

Đã chuẩn hóa, mã hóa và lưu thành công!
Kích thước dữ liệu mới: (7032, 37)


# Dữ liệu có đặc trưng mới được thêm vào 

In [42]:
# 1. Đọc dữ liệu từ file CSV gốc
df_preprocess = pd.read_csv('../data/feature_data/df_features.csv')

# Ép kiểu dữ liệu cho TotalCharges
df_preprocess['TotalCharges'] = pd.to_numeric(df_preprocess['TotalCharges'], errors='coerce')


In [43]:
# 1. Các cột số thuần túy (Không chứa Contract hay TenureGroup)
numeric_cols = [
    'SeniorCitizen',
    'tenure',
    'MonthlyCharges',
    'TotalCharges',
    'AverageMonthlyCharges',
    'ChargeRatio',
    'LifetimeValue_Proxy',
    'TotalServicesCount',
    'HasSecuritySuite',
    'HasStreamingSuite',
    'IsNewCustomer',
    'Contract_Tenure_Ratio',
    'IsNearContractEnd',
    'IsHighRiskPayment',
    'IsAutomaticPayment',
    'HouseholdSize_Proxy',
    'IsSingleSenior',
]

# 2. Cột Ordinal (đã mã hóa bằng OrdinalEncoder)
ordinal_cols = ['Contract', 'TenureGroup']

# 3. Cột Categorical còn lại (dùng OneHotEncoder)
categorical_cols = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'PaperlessBilling',
    'PaymentMethod',
]

exclude_cols = ['customerID', 'Churn']

In [44]:
# Khai báo thứ tự chuẩn xác cho từng cột ordinal
ordinal_encoder = OrdinalEncoder(categories=[
    ['Month-to-month', 'One year', 'Two year'],      # Thứ tự cho Contract (0, 1, 2)
    ['0-12', '12-24', '24-48', '>48']                # Thứ tự cho TenureGroup (0, 1, 2, 3)
])

# Áp dụng OrdinalEncoder vào dataframe
df_preprocess[ordinal_cols] = ordinal_encoder.fit_transform(df_preprocess[ordinal_cols])


ohe = OneHotEncoder(sparse_output=False, dtype=int, drop=None)
ohe_array = ohe.fit_transform(df_preprocess[categorical_cols])
ohe_feature_names = ohe.get_feature_names_out(categorical_cols)
df_ohe = pd.DataFrame(ohe_array, columns=ohe_feature_names, index=df_preprocess.index)



scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_preprocess[numeric_cols])
df_numeric = pd.DataFrame(scaled_array, columns=numeric_cols, index=df_preprocess.index)


# 5. GHÉP TẤT CẢ LẠI THÀNH DF_FINAL HOÀN CHỈNH (Đã bổ sung ordinal_cols)
df_final = pd.concat(
    [
        df_preprocess[exclude_cols],
        df_preprocess[ordinal_cols],
        df_numeric,
        df_ohe,
    ],
    axis=1,
)

print('Kích thước dataframe cuối cùng:', df_final.shape)

Kích thước dataframe cuối cùng: (7032, 59)


In [45]:
df_final.shape

(7032, 59)

In [46]:
df_final.head()

,customerID,Churn,Contract,TenureGroup,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,AverageMonthlyCharges,ChargeRatio,...,StreamingTV_Yes,StreamingMovies_No,StreamingMovies_No internet service,StreamingMovies_Yes,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,No,0.0,0.0,-0.440327,-1.280248,-1.161694,-0.994194,-1.157889,-0.044257,...,0,1,0,0,0,1,0,0,1,0
1,5575-GNVDE,No,1.0,2.0,-0.440327,0.064303,-0.260878,-0.173740,-0.305658,0.345250,...,0,1,0,0,1,0,0,0,0,1
2,3668-QPYBK,Yes,0.0,0.0,-0.440327,-1.239504,-0.363923,-0.959649,-0.355305,-0.044257,...,0,1,0,0,0,1,0,0,0,1
3,7795-CFOCW,No,1.0,2.0,-0.440327,0.512486,-0.747850,-0.195248,-0.791614,0.540004,...,0,1,0,0,1,0,1,0,0,0
4,9237-HQITU,Yes,0.0,0.0,-0.440327,-1.239504,0.196178,-0.940457,0.365282,-1.407534,...,0,1,0,0,0,1,0,0,1,0


In [47]:
# 6. Lưu toàn bộ tập dữ liệu thành file CSV vào thư mục đích
df_final.to_csv('../data/encoded/telco_churn_encoded.csv', index=False)

print("Đã chuẩn hóa, mã hóa và lưu thành công!")
print("Kích thước dữ liệu mới:", df_final.shape)

Đã chuẩn hóa, mã hóa và lưu thành công!
Kích thước dữ liệu mới: (7032, 59)
